# Uber Fare Prediction - Random Forest Regression

In this project, I am using a Random Forest Regressor to predict the fare of an Uber trip.

The model uses details such as trip distance, passenger count, trip duration, vehicle type, payment type, pickup time and locations.

At the end, I check the model using R2 score, MAE and RMSE and create a table with the actual and predicted fares.

## Import libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## Load data from MySQL

I am connecting to the MySQL database and reading the Uber trip table into a Pandas DataFrame.

In [ ]:
!pip install mysql-connector-python

import mysql.connector

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="root",
    database="uber"
)

df = pd.read_sql("SELECT * FROM uber_trip_details", conn)

print("Dataset shape:", df.shape)
df.head()

conn.close()

In [ ]:
df.head()


## Clean column names

In [ ]:
df.columns = df.columns.str.strip()

print(df.columns.tolist())

## Check Trip ID

I will need the Trip ID later when creating the prediction table.

In [ ]:
if "trip_id" not in df.columns:
    raise ValueError("Trip ID column was not found. Please check the column name.")

print("Trip ID column found.")

## Convert date and time columns

Convert the pickup and drop-off columns so that I can calculate trip duration and get time-based features.

In [ ]:
df["pickup_time"] = pd.to_datetime(df["pickup_time"], errors="coerce")
df["drop_off_time"] = pd.to_datetime(df["drop_off_time"], errors="coerce")

df[["pickup_time", "drop_off_time"]].head()

## Convert numeric columns

In [ ]:
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "surge_fee"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_columns].dtypes

## Create some new features

Here I am creating trip duration, pickup hour, pickup day and pickup month from the time columns.

In [ ]:
df["trip_duration"] = (
    df["drop_off_time"] - df["pickup_time"]
).dt.total_seconds() / 60

df["pickup_hour"] = df["pickup_time"].dt.hour
df["pickup_day"] = df["pickup_time"].dt.dayofweek
df["pickup_month"] = df["pickup_time"].dt.month

df[[
    "trip_duration",
    "pickup_hour",
    "pickup_day",
    "pickup_month"
]].head()

## Remove invalid data

In [ ]:
df = df[
    (df["fare_amount"] > 0) &
    (df["trip_distance"] > 0) &
    (df["trip_duration"] > 0) &
    (df["trip_duration"] < 300)
].copy()

print("Dataset shape after cleaning:", df.shape)

## Handle missing values

In [ ]:
df["passenger_count"] = df["passenger_count"].fillna(
    df["passenger_count"].median()
)

df["surge_fee"] = df["surge_fee"].fillna(0)

categorical_columns = [
    "vehicle",
    "payment_type",
    "pulocationid",
    "dolocationid"
]

for col in categorical_columns:
    df[col] = df[col].fillna("Unknown").astype(str)

print("Missing values handled.")

## Select features and target

In [ ]:
features = [
    "trip_distance",
    "passenger_count",
    "surge_fee",
    "trip_duration",
    "pickup_hour",
    "pickup_day",
    "pickup_month",
    "vehicle",
    "payment_type",
    "pulocationid",
    "dolocationid"
]

target = "fare_amount"

X = df[features]
y = df[target]

print("Number of features:", len(features))
print("Target:", target)

## Split the data

I am using 80% of the data for training and 20% for testing.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

## Separate numeric and categorical columns

In [ ]:
categorical_features = [
    "vehicle",
    "payment_type",
    "pulocationid",
    "dolocationid"
]

numeric_features = [
    "trip_distance",
    "passenger_count",
    "surge_fee",
    "trip_duration",
    "pickup_hour",
    "pickup_day",
    "pickup_month"
]

## One-hot encoding

The categorical columns need to be converted into numbers before training the model.

In [ ]:
preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("num", "passthrough", numeric_features)
])

## Random Forest model

In [ ]:
model = RandomForestRegressor(
    n_estimators=60,
    max_depth=15,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

## Build the pipeline

The pipeline handles the encoding and the Random Forest model together.

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", model)
])

## Train the model

In [ ]:
print("Training the model...")

pipeline.fit(X_train, y_train)

print("Model trained.")

## Make predictions

In [ ]:
y_pred = pipeline.predict(X_test)

print("Predictions are ready.")

## Check model performance

I am using R2, MAE and RMSE to see how well the model is performing.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("R2 Score:", round(r2, 4))
print("MAE:", round(mae, 2))
print("RMSE:", round(rmse, 2))

## Create a prediction table

In [ ]:
prediction_df = X_test.copy()

### Add Trip ID

In [ ]:
prediction_df.insert(
    0,
    "trip_id",
    df.loc[X_test.index, "trip_id"].values
)

### Add actual and predicted fare

In [ ]:
prediction_df["actual_fare"] = y_test.values
prediction_df["predicted_fare"] = y_pred

### Calculate prediction error

In [ ]:
prediction_df["prediction_error"] = (
    prediction_df["actual_fare"] - prediction_df["predicted_fare"]
)

prediction_df["absolute_error"] = prediction_df["prediction_error"].abs()

## View the predictions

In [ ]:
prediction_df[
    [
        "trip_id",
        "trip_distance",
        "passenger_count",
        "vehicle",
        "actual_fare",
        "predicted_fare",
        "prediction_error",
        "absolute_error"
    ]
].head(10)